In [9]:
# Import the libraries needed for file paths, dataframes, and optional SQL checks.
from pathlib import Path
import pandas as pd
import duckdb

# Define the curated data folder relative to this notebook.
DATA_DIR = Path("../data/curated")

# Define the datasets we want to QA.
files = {
    "da_2025": DATA_DIR / "da_prices_si_2025_clean.csv",
    "da_2026": DATA_DIR / "da_prices_si_clean.csv",
    "imbalance": DATA_DIR / "imbalance_si_clean.csv",
    "balancing": DATA_DIR / "balancing_volumes_si_clean.csv",
}

# Check that all expected files exist before loading them.
for name, file_path in files.items():
    print(name, "exists:", file_path.exists(), "|", file_path)

da_2025 exists: True | ..\data\curated\da_prices_si_2025_clean.csv
da_2026 exists: True | ..\data\curated\da_prices_si_clean.csv
imbalance exists: True | ..\data\curated\imbalance_si_clean.csv
balancing exists: True | ..\data\curated\balancing_volumes_si_clean.csv


### Day 22 QA context

This notebook performs data-quality checks on the curated Slovenian energy datasets.

The DA price data has an important market-structure feature: the Slovenian day-ahead Market Time Unit changed from 60-minute to 15-minute products from 2025-10-01 onward. This mixed resolution in the 2025 DA file is therefore expected and is not treated as a data-quality problem.

The QA checks focus on:
- timestamp coverage,
- duplicate timestamps,
- null values,
- missing intervals,
- DST transition behavior,
- value ranges,
- and known join mismatches between DA and imbalance data.

In [10]:
# Load each curated CSV into a separate pandas dataframe.
# parse_dates=["timestamp"] converts the timestamp column into proper datetime format.

datasets = {}

for name, file_path in files.items():
    df = pd.read_csv(file_path, parse_dates=["timestamp"])
    datasets[name] = df
    
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} columns")

da_2025: 15387 rows, 2 columns
da_2026: 8541 rows, 2 columns
imbalance: 11520 rows, 5 columns
balancing: 20160 rows, 66 columns


In [11]:
datasets["da_2025"]
datasets["da_2026"]
datasets["imbalance"]
datasets["balancing"]

,timestamp,Wpos MWh,Wneg MWh,Spos EUR,Sneg EUR,WpozFCR_Q' MWh,WnegFCR_Q' MWh,aRPF+akt (aFRR+act) MWh MWh,aFRR MWh,aFRR EUR,...,mFRR- akt SI>SI MWh MWh,QWpozVTL_NOS_Q MWh,QWnegVTL_NOS_Q MWh,QWpozVTL_HOPS_Q MWh,QWnegVTL_HOPS_Q MWh,QWpozVTL_PICASSO_Q MWh,QWnegVTL_PICASSO_Q MWh,QWpozVTL_MARI_Q MWh,QWnegVTL_MARI_Q MWh,settlement_type
0,2025-11-01 00:15:00,0.010,-20.263,1.54,-795.50,0.041,-0.002,0.010,0.010,1.54,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Second imbalance settlement
1,2025-11-01 00:30:00,0.200,-22.249,12.49,-798.75,0.011,-0.012,0.000,0.000,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Second imbalance settlement
2,2025-11-01 00:45:00,1.119,-10.075,61.84,-281.16,0.018,-0.012,0.000,0.000,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Second imbalance settlement
3,2025-11-01 01:00:00,3.795,-17.055,2.12,-785.39,0.005,-0.028,0.014,0.014,2.11,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Second imbalance settlement
4,2025-11-01 01:15:00,2.090,-20.921,139.35,-885.45,0.007,-0.009,0.143,0.018,2.58,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Second imbalance settlement
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20155,2026-02-28 23:30:00,0.963,-13.638,66.51,-427.16,0.000,0.000,0.000,0.000,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,First imbalance settlement
20156,2026-02-28 23:45:00,0.672,-8.529,32.03,-116.75,0.000,0.000,0.000,0.000,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,First imbalance settlement
20157,2026-02-28 23:45:00,0.672,-8.529,32.03,-116.75,0.000,0.000,0.000,0.000,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,First imbalance settlement
20158,2026-03-01 00:00:00,0.645,-6.832,3.02,-110.63,0.000,0.000,0.000,0.000,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,First imbalance settlement


In [12]:
# Show columns, data types, first timestamp, and last timestamp for each dataset.
# This tells us what each dataset contains and whether timestamp parsing worked correctly.

for name, df in datasets.items():
    print("=" * 80)
    print(name)
    print("- rows/columns:", df.shape)
    print("- columns:", list(df.columns))
    print("- dtypes:")
    print(df.dtypes)
    print("- first timestamp:", df["timestamp"].min())
    print("- last timestamp:", df["timestamp"].max())
    print("- first 3 rows:")
    display(df.head(3))

da_2025
- rows/columns: (15387, 2)
- columns: ['timestamp', 'price_eur_mwh']
- dtypes:
timestamp        datetime64[us]
price_eur_mwh           float64
dtype: object
- first timestamp: 2025-01-01 00:00:00
- last timestamp: 2026-01-01 00:00:00
- first 3 rows:


,timestamp,price_eur_mwh
0,2025-01-01 00:00:00,118.46
1,2025-01-01 01:00:00,129.07
2,2025-01-01 02:00:00,121.10


da_2026
- rows/columns: (8541, 2)
- columns: ['timestamp', 'price_eur_mwh']
- dtypes:
timestamp        datetime64[us]
price_eur_mwh           float64
dtype: object
- first timestamp: 2026-01-01 00:00:00
- last timestamp: 2026-04-01 00:00:00
- first 3 rows:


,timestamp,price_eur_mwh
0,2026-01-01 00:00:00,100.51
1,2026-01-01 00:15:00,94.03
2,2026-01-01 00:30:00,92.01


imbalance
- rows/columns: (11520, 5)
- columns: ['timestamp', 'imbalance_price_pos', 'imbalance_price_neg', 'sipx', 'settlement_type']
- dtypes:
timestamp              datetime64[us]
imbalance_price_pos           float64
imbalance_price_neg           float64
sipx                          float64
settlement_type                   str
dtype: object
- first timestamp: 2025-11-01 00:15:00
- last timestamp: 2026-03-01 00:00:00
- first 3 rows:


,timestamp,imbalance_price_pos,imbalance_price_neg,sipx,settlement_type
0,2025-11-01 00:15:00,20.71,20.71,103.63,Second imbalance settlement
1,2025-11-01 00:30:00,20.33,20.33,106.03,Second imbalance settlement
2,2025-11-01 00:45:00,20.94,20.94,85.55,Second imbalance settlement


balancing
- rows/columns: (20160, 66)
- columns: ['timestamp', 'Wpos MWh', 'Wneg MWh', 'Spos EUR', 'Sneg EUR', "WpozFCR_Q' MWh", "WnegFCR_Q' MWh", 'aRPF+akt (aFRR+act) MWh MWh', 'aFRR MWh', 'aFRR EUR', 'aFRR EUR/MWh', 'aRPF-akt (aFRR-act) MWh MWh', 'naFRR MWh', 'naFRR EUR', 'naFRR EUR/MWh', 'mFRR MWh', 'mFRR EUR', 'mFRR EUR/MWh', 'nmFRR  MWh', 'nmFRR  EUR', 'nmFRR  EUR/MWh', 'RR MWh', 'RR EUR', 'RR EUR/MWh', 'nRR MWh', 'nRR EUR', 'nRR EUR/MWh', 'IGCC (pos) MWh', 'IGCC (pos) EUR', 'IGCC (pos) EUR/MWh', 'IGCC (neg) MWh', 'IGCC (neg) EUR', 'IGCC (neg) EUR/MWh', 'VoAA+ EUR/MWh', 'VoAA- EUR/MWh', 'WnegFSkar MWh', 'FSkar (pos) MWh', 'SnegFSkar EUR', 'SpozFSkar EUR', 'Inbalance BS W+ MWh', 'Inbalance BS W- MWh', 'aFRR+ akt TUJ>SI EUR EUR', 'aFRR- akt TUJ>SI EUR EUR', 'aFRR+ akt TUJ>SII MWh MWh', 'aFRR- akt TUJ>SI MWh MWh', 'aFRR+ akt SI>SI EUR EUR', 'aFRR- akt SI>SI EUR EUR', 'aFRR+ akt SI>SI MWh MWh', 'aFRR- akt SI>SI MWh MWh', 'mFRR+ akt TUJ>SI  EUR EUR', 'mFRR- akt TUJ>SI  EUR EUR', 'mFRR+

,timestamp,Wpos MWh,Wneg MWh,Spos EUR,Sneg EUR,WpozFCR_Q' MWh,WnegFCR_Q' MWh,aRPF+akt (aFRR+act) MWh MWh,aFRR MWh,aFRR EUR,...,mFRR- akt SI>SI MWh MWh,QWpozVTL_NOS_Q MWh,QWnegVTL_NOS_Q MWh,QWpozVTL_HOPS_Q MWh,QWnegVTL_HOPS_Q MWh,QWpozVTL_PICASSO_Q MWh,QWnegVTL_PICASSO_Q MWh,QWpozVTL_MARI_Q MWh,QWnegVTL_MARI_Q MWh,settlement_type
0,2025-11-01 00:15:00,0.010,-20.263,1.54,-795.50,0.041,-0.002,0.01,0.01,1.54,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Second imbalance settlement
1,2025-11-01 00:30:00,0.200,-22.249,12.49,-798.75,0.011,-0.012,0.00,0.00,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Second imbalance settlement
2,2025-11-01 00:45:00,1.119,-10.075,61.84,-281.16,0.018,-0.012,0.00,0.00,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Second imbalance settlement


In [13]:
# Create a compact QA summary for each loaded dataset.
# This shows basic coverage, duplicate timestamps, and total missing values.

coverage_summary = []

for name, df in datasets.items():
    coverage_summary.append({
        "dataset": name,
        "rows": len(df),
        "columns": len(df.columns),
        "first_timestamp": df["timestamp"].min(),
        "last_timestamp": df["timestamp"].max(),
        "duplicate_timestamps": df["timestamp"].duplicated().sum(),
        "total_null_values": df.isna().sum().sum()
    })

coverage_summary_df = pd.DataFrame(coverage_summary)

display(coverage_summary_df)

,dataset,rows,columns,first_timestamp,last_timestamp,duplicate_timestamps,total_null_values
0,da_2025,15387,2,2025-01-01 00:00:00,2026-01-01,4,0
1,da_2026,8541,2,2026-01-01 00:00:00,2026-04-01,0,0
2,imbalance,11520,5,2025-11-01 00:15:00,2026-03-01,0,0
3,balancing,20160,66,2025-11-01 00:15:00,2026-03-01,8640,0


### Initial QA coverage summary

The curated datasets were loaded successfully and the timestamp column was parsed as datetime in all files.  
This first QA table checks the basic structure of each dataset: row count, column count, time coverage, duplicate timestamps, and total missing values.

This does not yet prove that the datasets are complete. It only confirms that the files are readable and structurally usable for deeper QA checks.

In [14]:
# Inspect duplicate timestamps in each dataset.
# This helps us understand whether duplicates are true data errors or expected market/reporting structure.

for name, df in datasets.items():
    duplicate_rows = df[df["timestamp"].duplicated(keep=False)].sort_values("timestamp")
    
    print("=" * 80)
    print(name)
    print("Duplicate timestamp rows:", len(duplicate_rows))
    
    if len(duplicate_rows) > 0:
        display(duplicate_rows.head(20))

da_2025
Duplicate timestamp rows: 8


,timestamp,price_eur_mwh
8959,2025-10-26 02:00:00,75.33
8963,2025-10-26 02:00:00,63.05
8960,2025-10-26 02:15:00,67.41
8964,2025-10-26 02:15:00,65.54
8961,2025-10-26 02:30:00,60.01
8965,2025-10-26 02:30:00,63.93
8962,2025-10-26 02:45:00,53.67
8966,2025-10-26 02:45:00,58.91


da_2026
Duplicate timestamp rows: 0
imbalance
Duplicate timestamp rows: 0
balancing
Duplicate timestamp rows: 17280


,timestamp,Wpos MWh,Wneg MWh,Spos EUR,Sneg EUR,WpozFCR_Q' MWh,WnegFCR_Q' MWh,aRPF+akt (aFRR+act) MWh MWh,aFRR MWh,aFRR EUR,...,mFRR- akt SI>SI MWh MWh,QWpozVTL_NOS_Q MWh,QWnegVTL_NOS_Q MWh,QWpozVTL_HOPS_Q MWh,QWnegVTL_HOPS_Q MWh,QWpozVTL_PICASSO_Q MWh,QWnegVTL_PICASSO_Q MWh,QWpozVTL_MARI_Q MWh,QWnegVTL_MARI_Q MWh,settlement_type
2880,2025-12-01 00:15:00,2.287,-18.030,208.94,-633.84,0.0,0.0,0.000,0.000,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Second imbalance settlement
2881,2025-12-01 00:15:00,2.287,-18.030,208.94,-633.84,0.0,0.0,0.000,0.000,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Second imbalance settlement
2882,2025-12-01 00:30:00,2.405,-3.077,129.54,-228.36,0.0,0.0,0.065,0.065,10.07,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Second imbalance settlement
2883,2025-12-01 00:30:00,2.405,-3.077,129.54,-228.36,0.0,0.0,0.065,0.065,10.07,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Second imbalance settlement
2884,2025-12-01 00:45:00,1.103,-4.410,64.00,-261.33,0.0,0.0,0.100,0.100,15.45,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Second imbalance settlement
2885,2025-12-01 00:45:00,1.103,-4.410,64.00,-261.33,0.0,0.0,0.100,0.100,15.45,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Second imbalance settlement
2886,2025-12-01 01:00:00,1.455,-4.339,122.34,-185.75,0.0,0.0,0.000,0.000,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Second imbalance settlement
2887,2025-12-01 01:00:00,1.455,-4.339,122.34,-185.75,0.0,0.0,0.000,0.000,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Second imbalance settlement
2889,2025-12-01 01:15:00,4.080,-4.044,238.89,-181.92,0.0,0.0,0.000,0.000,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Second imbalance settlement
2888,2025-12-01 01:15:00,4.080,-4.044,238.89,-181.92,0.0,0.0,0.000,0.000,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Second imbalance settlement


In [15]:
# Inspect duplicate timestamps only in datasets where the coverage summary found duplicates.
# This helps us understand whether duplicates are real data problems or expected DST/settlement effects.

datasets_with_duplicates = ["da_2025", "balancing"]

for name in datasets_with_duplicates:
    df = datasets[name]
    
    duplicate_rows = (
        df[df["timestamp"].duplicated(keep=False)]
        .sort_values("timestamp")
        .reset_index(drop=True)
    )
    
    print("=" * 80)
    print(name)
    print("Duplicate timestamp rows:", len(duplicate_rows))
    print("Unique duplicated timestamps:", duplicate_rows["timestamp"].nunique())
    
    display(duplicate_rows.head(30))

da_2025
Duplicate timestamp rows: 8
Unique duplicated timestamps: 4


,timestamp,price_eur_mwh
0,2025-10-26 02:00:00,75.33
1,2025-10-26 02:00:00,63.05
2,2025-10-26 02:15:00,67.41
3,2025-10-26 02:15:00,65.54
4,2025-10-26 02:30:00,60.01
5,2025-10-26 02:30:00,63.93
6,2025-10-26 02:45:00,53.67
7,2025-10-26 02:45:00,58.91


balancing
Duplicate timestamp rows: 17280
Unique duplicated timestamps: 8640


,timestamp,Wpos MWh,Wneg MWh,Spos EUR,Sneg EUR,WpozFCR_Q' MWh,WnegFCR_Q' MWh,aRPF+akt (aFRR+act) MWh MWh,aFRR MWh,aFRR EUR,...,mFRR- akt SI>SI MWh MWh,QWpozVTL_NOS_Q MWh,QWnegVTL_NOS_Q MWh,QWpozVTL_HOPS_Q MWh,QWnegVTL_HOPS_Q MWh,QWpozVTL_PICASSO_Q MWh,QWnegVTL_PICASSO_Q MWh,QWpozVTL_MARI_Q MWh,QWnegVTL_MARI_Q MWh,settlement_type
0,2025-12-01 00:15:00,2.287,-18.030,208.94,-633.84,0.0,0.0,0.000,0.000,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Second imbalance settlement
1,2025-12-01 00:15:00,2.287,-18.030,208.94,-633.84,0.0,0.0,0.000,0.000,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Second imbalance settlement
2,2025-12-01 00:30:00,2.405,-3.077,129.54,-228.36,0.0,0.0,0.065,0.065,10.07,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Second imbalance settlement
3,2025-12-01 00:30:00,2.405,-3.077,129.54,-228.36,0.0,0.0,0.065,0.065,10.07,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Second imbalance settlement
4,2025-12-01 00:45:00,1.103,-4.410,64.00,-261.33,0.0,0.0,0.100,0.100,15.45,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Second imbalance settlement
5,2025-12-01 00:45:00,1.103,-4.410,64.00,-261.33,0.0,0.0,0.100,0.100,15.45,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Second imbalance settlement
6,2025-12-01 01:00:00,1.455,-4.339,122.34,-185.75,0.0,0.0,0.000,0.000,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Second imbalance settlement
7,2025-12-01 01:00:00,1.455,-4.339,122.34,-185.75,0.0,0.0,0.000,0.000,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Second imbalance settlement
8,2025-12-01 01:15:00,4.080,-4.044,238.89,-181.92,0.0,0.0,0.000,0.000,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Second imbalance settlement
9,2025-12-01 01:15:00,4.080,-4.044,238.89,-181.92,0.0,0.0,0.000,0.000,0.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Second imbalance settlement


In [16]:
# Inspect the spring DST transition day in DA 2025.
# On 2025-03-30, clocks moved forward from 02:00 to 03:00.
# Because DA 2025 is hourly in March, we expect 23 hourly rows instead of 24.

da_2025 = datasets["da_2025"].copy()

spring_dst_day = pd.Timestamp("2025-03-30").date()

spring_dst_rows = (
    da_2025[da_2025["timestamp"].dt.date == spring_dst_day]
    .sort_values("timestamp")
    .reset_index(drop=True)
)

print("Spring DST day:", spring_dst_day)
print("Number of rows:", len(spring_dst_rows))

display(spring_dst_rows)

Spring DST day: 2025-03-30
Number of rows: 23


,timestamp,price_eur_mwh
0,2025-03-30 00:00:00,46.22
1,2025-03-30 01:00:00,15.88
2,2025-03-30 03:00:00,5.09
3,2025-03-30 04:00:00,1.20
4,2025-03-30 05:00:00,0.09
5,2025-03-30 06:00:00,0.93
6,2025-03-30 07:00:00,0.99
7,2025-03-30 08:00:00,1.45
8,2025-03-30 09:00:00,0.09
9,2025-03-30 10:00:00,-0.76


### Spring DST timestamp check — 2025-03-30

The DA 2025 dataset was checked around the spring DST transition.  
On 2025-03-30, Slovenia moved from winter time to summer time, with the clock moving forward from 02:00 to 03:00.

Because the DA 2025 dataset is hourly in March 2025, the expected count for this day is 23 intervals instead of 24.  
The timestamp inspection confirms that 02:00 is missing, which is expected for the spring DST transition and is not a data-quality problem.

In [17]:
# Check duplicate timestamps and duplicate timestamp + settlement_type combinations.
# This helps separate true data-quality problems from expected settlement-version structure.

duplicate_summary = []

for name, df in datasets.items():
    row = {
        "dataset": name,
        "rows": len(df),
        "duplicate_timestamps": df["timestamp"].duplicated().sum()
    }
    
    if "settlement_type" in df.columns:
        row["duplicate_timestamp_settlement_type"] = df.duplicated(
            subset=["timestamp", "settlement_type"]
        ).sum()
    else:
        row["duplicate_timestamp_settlement_type"] = None
    
    duplicate_summary.append(row)

duplicate_summary_df = pd.DataFrame(duplicate_summary)

display(duplicate_summary_df)

,dataset,rows,duplicate_timestamps,duplicate_timestamp_settlement_type
0,da_2025,15387,4,NaN
1,da_2026,8541,0,NaN
2,imbalance,11520,0,0.0
3,balancing,20160,8640,8640.0


In [18]:
# Show settlement type counts for Borzen datasets.
# This confirms how First and Second imbalance settlement rows are represented.

for name in ["imbalance", "balancing"]:
    df = datasets[name]
    
    print("=" * 80)
    print(name)
    
    if "settlement_type" in df.columns:
        settlement_counts = (
            df["settlement_type"]
            .value_counts(dropna=False)
            .rename_axis("settlement_type")
            .reset_index(name="rows")
        )
        
        display(settlement_counts)
    else:
        print("No settlement_type column found.")

imbalance


,settlement_type,rows
0,Second imbalance settlement,8832
1,First imbalance settlement,2688


balancing


,settlement_type,rows
0,Second imbalance settlement,14784
1,First imbalance settlement,5376


### Settlement type structure

The Borzen datasets contain a `settlement_type` column. Because of this, timestamp alone is not always the correct uniqueness key. For settlement-based datasets, the combination of `timestamp` and `settlement_type` is more informative.

The switch from Second to First imbalance settlement occurred on 2026-02-01. Duplicate timestamps in the balancing dataset should therefore be interpreted carefully and checked against settlement type before being treated as data errors.

In [19]:
# Combine DA 2025 and DA 2026 data for comparison with imbalance data.
# We drop duplicate DA timestamps so each timestamp appears only once in the join check.

da_all = pd.concat(
    [
        datasets["da_2025"].assign(source_file="da_prices_si_2025_clean.csv"),
        datasets["da_2026"].assign(source_file="da_prices_si_clean.csv"),
    ],
    ignore_index=True
)

da_all = da_all.sort_values("timestamp").reset_index(drop=True)

da_unique = da_all.drop_duplicates(subset=["timestamp"], keep="last")

imbalance = datasets["imbalance"].copy()

# Find imbalance timestamps that do not exist in DA data.
unmatched_imbalance = imbalance[
    ~imbalance["timestamp"].isin(da_unique["timestamp"])
].copy()

print("DA unique rows:", len(da_unique))
print("Imbalance rows:", len(imbalance))
print("Unmatched imbalance rows:", len(unmatched_imbalance))

display(unmatched_imbalance)

DA unique rows: 23923
Imbalance rows: 11520
Unmatched imbalance rows: 1


,timestamp,imbalance_price_pos,imbalance_price_neg,sipx,settlement_type
5759,2025-12-31,112.79,112.79,77.8,Second imbalance settlement


### DA + imbalance timestamp mismatch

The DA and imbalance datasets mostly align in the overlapping period.  
The previous Day 19 join matched 11,519 DA rows with imbalance rows.

The overlap contains:
- 11,519 DA rows,
- 11,520 imbalance rows,
- 11,519 joined rows.

The one unmatched imbalance row is caused by a missing DA timestamp. This is a real QA finding because it affects cross-dataset analysis. Any DA + imbalance analysis should either exclude this unmatched row or document that the join is based on matched timestamps only.

In [20]:
# Check important numeric columns for negative values, zeros, and high spikes.
# This gives a first sanity check on whether values are plausible.

value_checks = []

columns_to_check = {
    "da_2025": ["price_eur_mwh"],
    "da_2026": ["price_eur_mwh"],
    "imbalance": ["imbalance_price_pos", "imbalance_price_neg", "sipx"],
    "balancing": ["Wpos MWh", "Wneg MWh", "Spos EUR", "Sneg EUR"]
}

for dataset_name, columns in columns_to_check.items():
    df = datasets[dataset_name]
    
    for col in columns:
        if col in df.columns:
            value_checks.append({
                "dataset": dataset_name,
                "column": col,
                "min": df[col].min(),
                "max": df[col].max(),
                "mean": df[col].mean(),
                "negative_count": (df[col] < 0).sum(),
                "zero_count": (df[col] == 0).sum(),
                "above_500_count": (df[col] > 500).sum()
            })

value_checks_df = pd.DataFrame(value_checks)

display(value_checks_df)

,dataset,column,min,max,mean,negative_count,zero_count,above_500_count
0,da_2025,price_eur_mwh,-126.460,561.750,108.266467,269,62,5
1,da_2026,price_eur_mwh,-69.920,400.760,121.055885,84,12,0
2,imbalance,imbalance_price_pos,-3285.020,2745.440,121.935337,204,5,36
3,imbalance,imbalance_price_neg,-3285.020,2745.440,121.935337,204,5,36
4,imbalance,sipx,-0.440,400.760,120.018549,14,5,0
5,balancing,Wpos MWh,0.000,135.782,10.144370,0,113,0
6,balancing,Wneg MWh,-124.446,0.000,-10.344734,20088,72,0
7,balancing,Spos EUR,-719.400,155317.860,1402.573758,103,251,9708
8,balancing,Sneg EUR,-24020.450,79004.810,-668.631251,19718,155,129


### Value range check

The value range check flags negative values, zero values, and values above 500.

For DA and imbalance prices, negative prices and price spikes may be valid market outcomes, so they are not automatically treated as errors. They are flagged for awareness and may require later outlier analysis.

For balancing volumes and costs, zeros are common and usually represent intervals without activation or cost in that category. Negative values may also be meaningful depending on direction, especially for negative balancing energy or settlement costs.

### Day 22 QA conclusion

The curated datasets are structurally usable for analysis. All loaded datasets have valid timestamp parsing and no null values.

Important QA findings:
- The 2025 DA file has mixed MTU resolution: 60-minute before 2025-10-01 and 15-minute from 2025-10-01 onward. This is expected and reflects the market-rule transition.
- DST handling must be resolution-aware. Normal hourly days have 24 intervals, normal 15-minute days have 96 intervals, spring DST days have fewer intervals, and autumn DST days may contain repeated local timestamps.
- DA 2025 contains duplicate timestamps, likely related to autumn DST local-time repetition.
- Borzen settlement datasets require careful uniqueness checks because `settlement_type` affects the interpretation of duplicate timestamps.
- The DA + imbalance overlap has one unmatched imbalance timestamp caused by a missing DA timestamp.
- The 2026-04-01 DA row is a partial boundary day and should not be treated as a complete delivery day.
- Value-range checks flag possible outliers, but negative prices, zeros, and spikes are not automatically errors in electricity-market data.

Overall, the data is suitable for continued analysis, but joins and daily summaries must remain resolution-aware, DST-aware, and explicit about incomplete boundary days.